# ViT-Ctr Phase 3: Full Training on Colab T4
运行前请确认Runtime → Change runtime type → GPU (T4)

In [ ]:
# 环境安装
!pip install h5py --quiet  # h5py通常预装，但确保可用

In [ ]:
# 挂载Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CRITICAL: 数据集可用性检查 — Wave 0 Gate
import os, h5py

# 修改为实际的Google Drive路径
GDRIVE_DATA_DIR = '/content/drive/MyDrive/ViT-Ctr-data'
REQUIRED_FILES = ['dithioester.h5', 'trithiocarbonate.h5', 'xanthate.h5', 'dithiocarbamate.h5']
MIN_SAMPLES_PER_FILE = 100_000  # 完整数据集每文件约250K，此处用100K作为检查下限

all_ok = True
for fname in REQUIRED_FILES:
    fpath = os.path.join(GDRIVE_DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[FAIL] File missing: {fpath}")
        all_ok = False
    else:
        with h5py.File(fpath, 'r') as f:
            n = f['fingerprints'].shape[0]
            if n < MIN_SAMPLES_PER_FILE:
                print(f"[WARN] {fname}: only {n} samples (need >= {MIN_SAMPLES_PER_FILE})")
                all_ok = False
            else:
                print(f"[ OK] {fname}: {n} samples")

if not all_ok:
    raise RuntimeError(
        "数据集不完整！训练中止。\n"
        "请先运行 Phase 2 大规模生成脚本并上传到 Google Drive。\n"
        f"目标路径: {GDRIVE_DATA_DIR}"
    )
print("\n✓ 数据集验证通过，可以开始训练")

In [ ]:
# 克隆代码库
# 如果在Colab上：克隆或挂载代码
# 方式A: 从GitHub克隆（推荐）
# !git clone https://github.com/YOUR_USERNAME/ViT-Ctr.git /content/ViT-Ctr
# %cd /content/ViT-Ctr

# 方式B: 手动上传src/目录（适合调试）
# 请根据实际情况选择其中一种
import sys
sys.path.insert(0, '/content/ViT-Ctr/src')  # 根据实际路径修改

In [ ]:
# 复制HDF5到本地SSD（提升I/O速度）
import shutil, os
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
for fname in REQUIRED_FILES:
    src = os.path.join(GDRIVE_DATA_DIR, fname)
    dst = os.path.join(LOCAL_DATA_DIR, fname)
    if not os.path.exists(dst):
        print(f"Copying {fname}...")
        shutil.copy(src, dst)
H5_PATHS = [os.path.join(LOCAL_DATA_DIR, f) for f in REQUIRED_FILES]
print("H5_PATHS:", H5_PATHS)

In [ ]:
# 训练配置
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# 训练超参数（锁定值）
BATCH_SIZE = 64        # D-02
LR = 3e-4              # D-03
MAX_EPOCHS = 200       # D-05 (早停会提前结束)
NUM_WORKERS = 2        # Colab T4 推荐
CHECKPOINT_DIR = '/content/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# 运行训练
# 调用 src/train.py 的 main 逻辑
# 在Colab中推荐直接运行命令行接口
!python /content/ViT-Ctr/src/train.py \
    --h5_dir {LOCAL_DATA_DIR} \
    --epochs {MAX_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --seed 42

In [ ]:
# 损失曲线可视化
# 读取训练日志（JSON格式由train.py保存）并绘制loss曲线
import json, matplotlib.pyplot as plt
log_path = os.path.join(CHECKPOINT_DIR, 'training_log.json')
if os.path.exists(log_path):
    with open(log_path) as f:
        log = json.load(f)
    epochs = [e['epoch'] for e in log]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [e['train_loss'] for e in log], label='Train')
    axes[0].plot(epochs, [e['val_loss'] for e in log], label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Weighted MSE Loss')
    axes[0].set_title('Total Loss'); axes[0].legend()
    axes[1].plot(epochs, [e['loss_ctr'] for e in log], label='Ctr (×2.0)')
    axes[1].plot(epochs, [e['loss_inh'] for e in log], label='Inhibition')
    axes[1].plot(epochs, [e['loss_ret'] for e in log], label='Retardation')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Per-output MSE')
    axes[1].set_title('Per-output Loss'); axes[1].legend()
    plt.tight_layout(); plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_curves.png'), dpi=150)
    plt.show()
else:
    print(f"Log file not found at {log_path}")

## 训练完成后
1. 将 `checkpoints/best_model.pth` 下载到本地 `checkpoints/` 目录
2. 记录最佳验证集loss和达到早停时的epoch数
3. 运行 `colab/03-bootstrap-colab.ipynb` 进行Bootstrap UQ